In [ ]:
import os
import sys
import gc
import json
import time
import warnings
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay, fbeta_score
import matplotlib.pyplot as plt
import seaborn as sns
from torch.optim import Adam 
from tabulate import tabulate

device = torch.device("cuda" if torch.cuda.is_available() else 
                     ("mps" if torch.backends.mps.is_available() else "cpu"))

def clear_memory():
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
sys.path.append(str(ROOT))

from Utils.project_utils import *
import Utils.constants as c
from Utils.transforms import *
from Utils.MushroomDataset import *
from Utils.dataLoaders import *
from Utils.lookahead import Lookahead

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print(f"Using device: {device}")

In [ ]:
SPLIT_DIR = Path.cwd().resolve() / "Data" / "splits"

In [ ]:
# Load the datasets
train = pd.read_csv(SPLIT_DIR / "train.csv")
val = pd.read_csv(SPLIT_DIR / "val.csv")
test = pd.read_csv(SPLIT_DIR / "test.csv")

In [ ]:
# Load the class mapping
with open(SPLIT_DIR / "class_to_idx.json", "r") as f:
    train_class_to_idx = json.load(f)

In [ ]:
# Define transforms (fixing the syntax error from before)
train_tfms_mild = get_train_tfms_mild() 
train_tfms_strong = get_train_tfms_strong() 
val_tfms = get_val_tfms()

print("All files loaded successfully!")

In [ ]:
save_dir = PROJECT_ROOT / "Graphs"
save_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
CLASSES = sorted(train["class"].unique().tolist())

In [ ]:
loaders = make_loaders(
    train,
    val,
    test,
    train_tfms_mild,
    train_tfms_strong,
    val_tfms,
    train_class_to_idx,
    CLASSES,
    c.BATCH_SIZE,
    c.NUM_WORKERS,
    device,
)

loaders

In [ ]:
train_loader = loaders["train_loader_strong"]
val_loader = loaders["val_loader"]
test_loader = loaders["test_loader"]

In [ ]:
toxic_idx = train_class_to_idx['toxic']

In [ ]:
class_weights = torch.tensor([1.0, 1.0, 8.0], dtype=torch.float32, device=device)

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)

## Training and Validation

### Load RESNET50 Model

In [ ]:
def run_one_epoch(model, loader, criterion, optimizer, device, is_train=False, return_preds=False, scheduler=None):
    running_loss, running_correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    
    model.train() if is_train else model.eval()
    num_batches = len(loader)
    
    for i, (img, labels) in enumerate(loader):
        img, labels = img.to(device), labels.to(device)

        with torch.set_grad_enabled(is_train):
            outputs = model(img)
            loss = criterion(outputs, labels)
            
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
                if scheduler: scheduler.step()

        preds = outputs.argmax(1)
        running_loss += loss.item() * img.size(0)
        running_correct += (preds == labels).sum().item()
        total += img.size(0)

        if (i + 1) % 5 == 0 or (i + 1) == num_batches:
            print(f"  {'Train' if is_train else 'Val'} [{i+1}/{num_batches}] Loss: {loss.item():.4f}", end='\r')

        if return_preds:
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

    print() 
    metrics = [running_loss / total, running_correct / total]

    if return_preds:
        y_true, y_pred = torch.cat(all_labels).numpy(), torch.cat(all_preds).numpy()
        
        # general Metrics
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        f1_m = f1_score(y_true, y_pred, average='macro')
        f1_w = f1_score(y_true, y_pred, average='weighted')
        
        # toxic Class Metrics (Uses your toxic_idx variable)
        # beta=2.0 calculates the F2 score (Priority) directly
        prec, rec, f2_score, _ = precision_recall_fscore_support(
            y_true, y_pred, labels=[toxic_idx], beta=2.0, zero_division=0
        )
        
        # standard f1 matrix for printout
        _, _, f1_tox, _ = precision_recall_fscore_support(
            y_true, y_pred, labels=[toxic_idx], beta=1.0, zero_division=0
        )
        
        # [bal_acc, f1_macro, f1_weighted, precision, recall, f1_toxic, priority_f2]
        metrics.extend([bal_acc, f1_m, f1_w, prec[0], rec[0], f1_tox[0], f2_score[0]])
    else:
        metrics.extend([None] * 7) 
    
    return tuple(metrics)

In [ ]:
BEST_MODEL_PATH = Path.cwd() / "best_models" / "resnet50_best.pth"

In [ ]:
BEST_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def train(model, train_loader, val_loader, criterion, optimizer, num_epochs, device, patience=5, scheduler=None):
    best_priority = -float("inf")
    bad_epochs = 0

    for epoch in range(num_epochs):
        # Training Dataset
        t_loss, t_acc, _, _, _, _, _, _, _ = run_one_epoch(
            model, train_loader, criterion, optimizer, device, is_train=True, scheduler=scheduler
        )
        
        # Validating Dataset
        v_loss, v_acc, v_bal, v_f1_m, v_f1_w, v_tox_p, v_tox_r, v_tox_f1, v_prio = run_one_epoch(
            model, val_loader, criterion, optimizer, device, is_train=False, return_preds=True
        )

        current_lr = optimizer.param_groups[-1]['lr']

        print(f"\n>>> EPOCH {epoch+1} RESULTS <<<")
        print(f"Loss: [Train: {t_loss:.4f} | Val: {v_loss:.4f}]")
        print(f"Acc:  [Train: {t_acc:.4f}  | Val: {v_acc:.4f}]")
        print(f"Overall: [Balanced Acc: {v_bal:.4f} | Macro F1: {v_f1_m:.4f} | Weighted F1: {v_f1_w:.4f}]")
        print(f"Toxic:   [Precision: {v_tox_p:.4f} | Recall: {v_tox_r:.4f} | F1: {v_tox_f1:.4f}]")
        print(f"Score:   [PRIORITY: {v_prio:.4f} | LR: {current_lr:.8f}]")

        if v_prio > best_priority:
            best_priority = v_prio
            bad_epochs = 0
            torch.save(model.state_dict(), BEST_MODEL_PATH)
        else:
            bad_epochs += 1
            print(f"--- No improvement. Patience: {bad_epochs}/{patience}")
            if bad_epochs >= patience:
                break
        
        clear_memory()

    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    return model

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
model = model.to(device);

In [ ]:
base_opt = torch.optim.RAdam([
    {"params": model.conv1.parameters(), "lr": 1e-5},
    {"params": model.bn1.parameters(), "lr": 1e-5},
    {"params": model.layer1.parameters(), "lr": 1e-5},
    {"params": model.layer2.parameters(), "lr": 2e-5},
    {"params": model.layer3.parameters(), "lr": 5e-5},
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3},
], weight_decay=0.01)

optimizer = Lookahead(base_opt, k=5, alpha=0.5)

In [ ]:
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    base_opt, 
    max_lr=[1e-5, 1e-5, 1e-5, 2e-5, 5e-5, 1e-4, 1e-3], 
    steps_per_epoch=len(train_loader), 
    epochs=30
)

In [ ]:
model = train(model, train_loader, val_loader, criterion, optimizer, num_epochs=30, device=device, patience=5, scheduler=scheduler)
model.eval(); 

In [ ]:
results = run_one_epoch(model, test_loader, criterion, None, device, is_train=False, return_preds=True)
t_loss, t_acc, t_bal, t_f1m, t_f1w, t_prec, t_rec, t_f1tox, t_priority = results


In [ ]:
print(f"Loss:      {t_loss:.4f}")
print(f"Accuracy:  {t_acc:.4f} (Balanced: {t_bal:.4f})")
print(f"F1 Scores: [Macro: {t_f1m:.4f} | Weighted: {t_f1w:.4f}]")

In [ ]:
print(f"\n TOXIC CLASS PERFORMANCE:")
print(f"  Precision: {t_prec:.4f}")
print(f"  Recall:    {t_rec:.4f} <--- (This is the most important for safety!)")
print(f"  F1 Score:  {t_f1tox:.4f}")
print(f"  PRIORITY (F2): {t_priority:.4f}")

In [ ]:
results = run_one_epoch(model, test_loader, criterion, None, device, is_train=False, return_preds=True)
test_loss, test_acc, _, _, _, _, _, _, test_f2 = results

print(f"Test Accuracy: {test_acc:.4f} | Test Toxic F2 (Priority): {test_f2:.4f}")

In [ ]:
torch.save(model.state_dict(), "Mushroom_ResNet50_Final.pth")
print("Successfully saved final optimized model weights!")

## Testing

In [ ]:
test_loss, test_acc, test_total, test_labels, test_preds, test_f2 = run_one_epoch(model, test_loader, criterion, None, device, is_train=False, return_preds=True)

In [ ]:
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test F2 score (macro): {test_f2:.4f}")

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
print("Confusion matrix:\n", cm)
print("\nClassification report:\n",
      classification_report(test_labels, test_preds, target_names=CLASSES, digits=4))

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
fig, ax = plt.subplots(figsize=(6,6))
disp.plot(ax=ax, cmap="Blues", colorbar=False)

plt.title("Confusion Matrix - ResNet50")
plt.tight_layout()

# 3. SAVE FIRST, THEN SHOW
plt.savefig(save_dir / "resnet50_confusion_matrix.png", dpi=300)
plt.show()